# Notebook 06: Results Summary & Discussion

## Abstract

This project reproduces and extends findings from Conti et al. (2025), which reported that CRISPR-Cas9/AAV6-mediated homology-directed repair (HDR) gene editing induces a p53-driven DNA damage response and IL-1/NF-κB-mediated inflammatory program in human hematopoietic stem and progenitor cells (HSPCs), and that the IL-1 receptor antagonist Anakinra mitigates this response.


### Part 1: Reproduction of Analysis:

Using a Python-based pipeline (pydeseq2, gseapy), I reproduced the paper's central finding that CRISPR-Cas9/AAV6 gene editing (GE) triggers a p53-driven DNA damage response and NF-κB-mediated inflammatory program in edited cells relative to unedited controls (GE vs. RNP_NEG). My results showed the correct direction and statistical significance at both 24h and 96h post-editing, across five specifically named Hallmark pathways:

- p53 Pathway
- TNF-alpha Signaling via NF-kB
- Myc Targets V1/V2
- E2F Targets

The reported Anakinra rescue effect (GE_ANAK vs. GE) was also directionally consistent (negative NES for all five target pathways at 96h). However, it did not reach statistical significance. At 24h, my results showed a partial discrepancy from the paper.
 
### Part 2: Extension:
I continued my analysis by building a rescue-percentage metric to quantify how far Anakinra treatment moves expression back toward the unedited baseline.

The data showed a real, timepoint-dependent rescue signal that wasn't visible from significance-based classification alone:
- Median rescue of approximately -1.8% at 24h
- Median rescue of approximately +8.7% at 96h

I followed up by testing whether this rescue is mechanistically selective. The data was inconclusive.

### Overall: 
This project demonstrates that the paper's claim is robustly reproducible with an independent pipeline. The secondary claim was also supported directionally, but statistical significance was not reproducible.


## 1. Reproduction Results

### 1.1 Editing effect (GE vs. RNP_NEG)

The main claim of the paper is that gene editing induces upregulation of DDR-related genes and inflammatory response genes, along with downregulation of c-Myc and E2F targets.

To test this, I ran GSEA on my own DESeq2 results, ranking genes by log2FoldChange (matching the paper's stated method), and checked the five pathways they specifically named:

| Pathway | NES (24h) | FDR (24h) | NES (96h) | FDR (96h) |
|---|---|---|---|---|
| p53 Pathway | +2.56 | <0.001 | +1.53 | 0.049 |
| TNF-alpha Signaling via NF-kB | +2.39 | <0.001 | +1.78 | 0.006 |
| Myc Targets V1 | -2.13 | <0.001 | -1.71 | 0.021 |
| Myc Targets V2 | -1.67 | 0.010 | -1.59 | 0.040 |
| E2F Targets | -2.34 | <0.001 | -1.76 | 0.024 |

**All five pathways reproduced with the correct direction and reached significance at both timepoints.**

Effect size and significance were strongest at 24h, weakening somewhat by 96h, but still remained significant. This tells me the transcriptional response to editing is present early and mostly persists over time, rather than building up gradually or only appearing later.

Effect size anf significance was strongest at 24, weakening by 96h, but still remained significant. 

My analysis was able to get the same five named pathways, correction direction, and statistical significance, using a completely different toolchain than the original paper (Python with pydeseq2 and gseapy, instead of their R pipeline with DESeq2 and clusterProfiler), along with independently made QC and filtering decisions. 

If there were an error in the pipeline itself, it would most likely have affected this result too, since this is the strongest and most reproducible signal in the dataset. Getting a clean match here supports confidence that the pipeline is working correctly. 

![GSEA summary, 24h](../results/figures/gsea_dotplot_24h_ge.png)
![GSEA summary, 96h](../results/figures/gsea_dotplot_96h_ge.png)

### 1.2 Anakinra rescue effect (GE_ANAK vs. GE)

The paper additionally reported that Anakinra treatment downregulates IFN-alpha/gamma response and NF-κB-related inflammatory pathways (IL-2/STAT5, IL-6/JAK/STAT3, TNF-α/NF-κB signaling) at both 24h and 96h timepoints.

The authors' own analysis code applies **no significance filter** to this specific comparison. All 35 named Hallmark terms were plotted by raw NES, regardless of FDR, unlike their primary comparison (GE vs. RNP_NEG), which was filtered to FDR<0.05.

Two checks were run:
- the five specifically named pathways
- the full 35-term unfiltered comparison

**Individual-gene level**:
- Zero genes reach significance (padj<0.05) in the GE_ANAK_vs_GE contrast at either timepoint.
- A sensitivity check confirmed this null result is not an artifact of the gene filtering step. 
    - Re-ran the same contrast on completely unfiltered counts still produced 0 significant genes at both timepoints.

**Pathway level: Five target pathways**:

| Pathway | NES (24h) | NES (96h) |
|---|---|---|
| Interferon Alpha Response | -0.77 | -0.86 |
| Interferon Gamma Response | **+0.70** | -0.87 |
| TNF-alpha Signaling via NF-kB | -1.02 | -0.91 |
| IL-2/STAT5 Signaling | -0.74 | -1.20 |
| IL-6/JAK/STAT3 Signaling | -0.88 | -1.04 |

- At 96h, **all five** pathways showed the expected negative NES.
- At 24h, four of five pathways showed the expected negative NES. Interferon Gamma Response showed a positive NES instead.
    - This is a discrepancy from the paper's claim of downregulation at both early and later timepoints.
- **Note**: none of these pathways reached FDR<0.05 in this analysis.

**Full 35-term Hallmark comparison** (no significance filter, matching the authors' own approach for this figure): 
- At 96h, 27 of 35 terms (77%) show negative NES
    - This broad downregulation result is consistent with the paper. 
- At 24h, the direction was nearly balanced (20 negative / 15 positive)
    - This does not show a clear directional pattern, though it could be argued to lean slightly negative.

**Summary**: 
The Anakinra rescue effect is directionally reproducible and strongest at 96h. It is not statistically significant at the individual-pathway level at this sample size (n=3/group), and shows a specific, honestly-reported discrepancy at 24h for one pathway. The weaker results in this section are more likely explained by factors like sample size, the specific comparison being tested, or differences in how the two comparisons were treated in the original paper (see the significance-filter note above), rather than a flaw in the pipeline itself.

## 2. Extension: Quantifying Rescue

### 2.1 Rescue-percentage metric

Expanding on the results above, a rescue percentage metric was built to quantify how far Anakinra treatment moves expression back toward the unedited baseline (RNP_NEG), specifically for genes significantly affected by editing.

**rescue % = (GE_log2FC − GE_ANAK_log2FC) / GE_log2FC × 100**

- **GE_log2FC** = log2 fold change of GE vs. RNP_NEG. This is how much expression changed from gene editing alone against unedited baseline. 

- **GE_ANAK_log2FC** = log2 fold change of GE_ANAK vs. RNP_NEG. This is how much the same gene's expression differ in the Anakinra-treated, edited cells against unedited baseline.  

| Timepoint | n genes | Median rescue % | Mean rescue % |
|---|---|---|---|
| 24h | 560 | -1.8% | -1.2% |
| 96h | 39 | **8.7%** | 12.1% |

![Rescue percentage distribution](../results/figures/rescue_percent_distribution.png)

**At 24h, the distribution is tightly centered near zero, showing no net rescue signal. At 96h, the distribution is visibly right-shifted, indicating a real, modest, timepoint-dependent rescue effect.**

### 2.2 Categorical rescue classification

A formal categorical rescue classification (full/partial/persistent) was also built, based on significance testing of GE_ANAK_vs_GE and GE_ANAK_vs_RNP_NEG. Since zero genes reached individual significance in GE_ANAK_vs_GE at either timepoint, every gene was classified as "persistent" at both timepoints.

This is not a contradiction of the rescue-percentage result above. It reflects a real limitation of significance-based classification at this sample size (n=3 per group). The rescue-percentage metric was able to detect a pattern that the categorical method, by design, could not.

### 2.3 Mechanism selectivity: DDR/p53 vs. inflammatory/NF-κB genes

Anakinra's rescue effect was tested for mechanistic selectivity. The original hypothesis assumed Anakinra, as an IL-1 receptor antagonist acting downstream of the DNA damage response, would rescue inflammatory/NF-κB genes while leaving DDR/p53 genes largely unaffected, on the assumption that these two programs act independently.

Using the GE vs. RNP_NEG GSEA results:
- 44 P53-pathway exclusive genes
- 60 TNF/NF-kB exclusive genes

17 genes were found to be shared between the two pathways. These were removed before computing rescue percentages. The remaining exclusive genes were then filtered to those with a stable, meaningful GE effect size (|log2FC|>1, required to avoid division-by-near-zero instability in the rescue percentage calculation). After this filtering, the number of genes remaining in each group was:
- 10 P53-only 
- 13 TNF-only genes

**Median rescue percentage was calculated:**
**- 20.4% for P53-only genes**
**- 2.3% TNF-only genes**

**A Mann-Whitney U test found no significant difference (p=0.31).**


This result is inconclusive. It does not confirm the original hypothesis, but it also does not disprove it. The small sample remaining after the effect-size filter (n=10, n=13) is substantially underpowered to detect a real difference even if one exists.

**Note:** the original hypothesis assumed p53 and NF-κB signaling act independently. However, it is documented that NF-κB signaling can reinforce and prolong p53 activation dynamics. If Anakinra disrupts this feedback loop, it could indirectly reduce the NF-κB-reinforced component of the p53 signature over time, not just NF-κB genes directly. This offers one possible explanation for the direction seen here, but it remains untested with this dataset and should not be treated as confirmed.

## 3. Discussion

This reproduction demonstrated that Conti et al. (2025)'s claim that CRISPR-Cas9/AAV6 HDR editing induces a p53/DDR and NF-κB/inflammatory transcriptional response in HSPCs is robust and independently reproducible, using a different implementation stack, independent QC, and filtering decisions.

The secondary claim that Anakinra rescues this response is more nuanced. This re-analysis supports the direction of the claimed effect at the pathway level, particularly at 96h, but it did not reach statistical significance for any individual pathway or gene at this sample size.

Extending this analysis, a rescue percentage metric was computed to reveal a quantifiable, timepoint-dependent rescue signal that a standard significance-based classification can miss. This shows that at small sample sizes, categorical significance testing and effect-size quantification can diverge substantially, and relying on either one alone risks overstating or understating the true result.

The mechanism-selectivity test (DDR vs. inflammatory gene rescue) remains an open question. A definitive answer would require either a larger dataset or a more numerically stable variant of the rescue metric that doesn't require discarding most candidate genes to a strict effect-size filter.


## 4. Limitations

- **Small sample size** (n=3 biological replicates per condition per timepoint) limits statistical power throughout, particularly for the Anakinra rescue effect and the mechanism-selectivity test.

- **24h and 96h samples were sequenced with different library preparation protocols** (paired-end stranded vs. single-end unstranded, per the original paper's methods), confounding any direct comparison of timepoint as a biological variable with timepoint as a technical/protocol variable. This was addressed by fitting fully separate models per timepoint rather than a combined interaction model, but cross-timepoint comparisons (is the effect stronger at 96h over 24h) should still be interpreted with this caveat in mind.

- **Senescence-specific gene sets** used in the original paper (SenMayo, Purcell, Pribluda_SENESCENCE_INFLAMMATORY_GENES, Sencan, SASP_SCHLEICH) were not incorporated into this reproduction. Of these, only SenMayo was found in a public database (MSigDB), and only as a mouse-gene-symbol version. A human-symbol equivalent was not confirmed. The other four could not be located as standalone, downloadable gene sets. The core reproduction is unaffected, as it relies on the independently-confirmed Hallmark pathway results.

- **The rescue-percentage metric is numerically unstable for genes with a small denominator** (small GE effect size). This was addressed with an effect-size filter for the mechanism-split analysis specifically, which substantially reduced the usable sample size for that analysis.

- **One PCA outlier per timepoint** (E2 at 24h, N7 at 96h) was identified and retained rather than excluded, since removing either would reduce its condition group to n=2. E2's outlier status is explained by unusually low library size; N7's is not explained by any diagnostic checked here.

- This is a **single independent reproduction** of one paper's dataset. It does not constitute independent biological replication (a new experiment), only independent computational reanalysis of the same underlying data.